# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SafeeAkmal/FlyRank-ML-internship-tasks/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Ans: **Method: Random Forest Regressor**, predicting `ctr_30d` from position, volume,
and content features. Ranking output = actual CTR minus model-predicted
(expected) CTR — pages with the most negative residual are the biggest
opportunities, replacing the Week 4 rule's single-variable (position-tier-only)
comparison with a multi-feature one.

**Why this method fits Lane 4:**

Position has a clearly non-linear relationship with CTR — the marginal drop
in CTR accelerates as position worsens (roughly 10-15% relative decline per
bucket near the top of the page, versus 40-45% per bucket beyond position 20).
A linear model (e.g. Logistic Regression) would misrepresent both ends of
this curve, systematically under- or over-penalizing pages depending on where
they sit. Tree-based ensembles capture this shape directly through splits,
with no manual transformation needed.

Clustering was ruled out because the task isn't to discover unlabeled groups
of pages — it's to measure a specific quantity (expected CTR vs actual) for
each page, which needs a regression target, not a grouping. A single Decision
Tree was ruled out in favor of an ensemble (Random Forest) because a lone tree
is high-variance and unstable — a bad fit given that Week 4 already found the
underlying data contains a known contamination pattern concentrated in
specific clients; an ensemble average is far less likely to be dominated by
one anomalous split the way a single tree, or the Week 4 rule's raw
`impressions x gap` score, was dominated by a handful of corrupted rows.

**Two dataset properties, confirmed by direct check rather than assumed,
inform how the model is built:**

1. CTR values in this dataset are structurally far below real-world search
   benchmarks (median ~0.24-0.29% at position <=3, even at 1000+ impressions,
   versus a typical real-world 20-30%). Ruled out small-sample noise as the
   cause — volume-stratified check showed median CTR stabilizes once
   impressions_30d exceeds ~200 and does not climb further with more volume.
   This means CTR here is usable only in relative/directional terms within
   this dataset, never compared to outside industry benchmarks.
2. Because of (1), and consistent with the Week 4 signal check, only pages
   with impressions_30d >= 200 are used for training and evaluation — this
   removes the noisiest rows (mean/median diverge sharply below this
   threshold, indicating single-click flukes), not because it reveals a
   "true" higher CTR underneath.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, pandas as pd, numpy as np

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
from google.colab import userdata
hf_token = userdata.get('hf_token')
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}')")

MONTH = "2026-03"
fact_path    = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/data_0.parquet"
content_path = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# Apply last week's fix up front: exclude impossible-position days before aggregating
df = con.execute(f"""
    WITH clean AS (
        SELECT * FROM read_parquet('{fact_path}')
        WHERE gsc_data_available = TRUE AND gsc_avg_position >= 1.0
    ),
    agg AS (
        SELECT content_hash_id, client_hash_id,
            SUM(gsc_impressions) AS impressions_30d,
            SUM(gsc_clicks) AS clicks_30d,
            SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions),0) AS avg_position_30d,
            SUM(ga4_sessions) AS sessions_30d,
            SUM(ga4_engaged_sessions) AS engaged_sessions_30d
        FROM clean
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT a.*, c.content_type, c.main_intent, c.word_count, c.is_published, c.is_deleted
    FROM agg a
    JOIN read_parquet('{content_path}') c USING (content_hash_id)
    WHERE c.is_published = TRUE AND c.is_deleted = FALSE
      AND a.impressions_30d >= 200
""").df()

df["ctr_30d"] = df["clicks_30d"] / df["impressions_30d"]

# Bin position into fine buckets and check whether the CTR-position relationship is linear
df["pos_bin"] = pd.cut(df["avg_position_30d"], bins=[0,1,2,3,5,8,12,20,30,50,100])
print(df.groupby("pos_bin")["ctr_30d"].mean())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pos_bin
(0, 1]            NaN
(1, 2]       0.004175
(2, 3]       0.003993
(3, 5]       0.003535
(5, 8]       0.003079
(8, 12]      0.002686
(12, 20]     0.002423
(20, 30]     0.001719
(30, 50]     0.000982
(50, 100]    0.000406
Name: ctr_30d, dtype: float64


/tmp/ipykernel_1332/560776591.py:42: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("pos_bin")["ctr_30d"].mean())


In [3]:
# Check whether this magnitude issue is dataset-wide or specific to the
# impressions_30d >= 200 filter (e.g. maybe you're only seeing high-impression,
# low-CTR long-tail queries after that cutoff)
print("Without the impressions filter:")
df_unfiltered = con.execute(f"""
    WITH clean AS (
        SELECT * FROM read_parquet('{fact_path}')
        WHERE gsc_data_available = TRUE AND gsc_avg_position >= 1.0
    ),
    agg AS (
        SELECT content_hash_id,
            SUM(gsc_impressions) AS impressions_30d,
            SUM(gsc_clicks) AS clicks_30d,
            SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions),0) AS avg_position_30d
        FROM clean GROUP BY content_hash_id
    )
    SELECT * FROM agg WHERE impressions_30d > 0
""").df()
df_unfiltered["ctr_30d"] = df_unfiltered["clicks_30d"] / df_unfiltered["impressions_30d"]
print(df_unfiltered[df_unfiltered["avg_position_30d"] <= 3]["ctr_30d"].describe())

Without the impressions filter:
count    12373.000000
mean         0.011513
std          0.081590
min          0.000000
25%          0.000000
50%          0.000695
75%          0.003784
max          1.000000
Name: ctr_30d, dtype: float64


In [4]:
df_unfiltered["vol_bucket"] = pd.cut(
    df_unfiltered["impressions_30d"],
    bins=[0, 10, 50, 200, 1000, 1e9],
    labels=["1-10", "11-50", "51-200", "201-1000", "1000+"]
)

pos3 = df_unfiltered[df_unfiltered["avg_position_30d"] <= 3]
print(pos3.groupby("vol_bucket", observed=True)["ctr_30d"].agg(["count", "mean", "median"]))

            count      mean    median
vol_bucket                           
1-10         4193  0.025309  0.000000
11-50         540  0.008865  0.000000
51-200        791  0.004982  0.000000
201-1000     1873  0.003717  0.002367
1000+        4976  0.004147  0.002919


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Ans:  GroupKFold split by client_hash_id, not a random row
split. Week 4 found the impossible-position defect concentrates heavily in
specific clients; a random split would let the same client's rows appear in
both train and test, letting the model partly memorize client-specific
quirks and inflating its apparent accuracy. Grouping by client forces the
model to prove it generalizes to clients it has never seen.
Row-count check across the 5 GroupKFold folds shows reasonably even test
splits (19.5%-22.0% of rows per fold), despite uneven client counts per fold
(1 to 12 test clients). This means the apparent imbalance in client count
(Fold 0: only 1 test client) does not translate into an imbalanced amount of
data — that single client is simply large. Per-fold scores are still reported
individually rather than only averaged, since testing on a single (even if
large) client remains a narrower generalization test than testing on 9-12
clients, and any large gap between Fold 0's score and the others would be
worth flagging on its own.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
groups = df["client_hash_id"]

# Show that a naive random split would leak clients across folds; GroupKFold does not
for fold, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
    train_clients = set(df.iloc[train_idx]["client_hash_id"])
    test_clients  = set(df.iloc[test_idx]["client_hash_id"])
    overlap = train_clients & test_clients
    print(f"Fold {fold}: train clients={len(train_clients)}, test clients={len(test_clients)}, overlap={len(overlap)}")

Fold 0: train clients=41, test clients=1, overlap=0
Fold 1: train clients=33, test clients=9, overlap=0
Fold 2: train clients=32, test clients=10, overlap=0
Fold 3: train clients=30, test clients=12, overlap=0
Fold 4: train clients=32, test clients=10, overlap=0


In [6]:
# Confirm row-count balance across folds, not just client-count balance
for fold, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
    print(f"Fold {fold}: train rows={len(train_idx)}, test rows={len(test_idx)}, "
          f"test_pct={len(test_idx)/len(df):.1%}")

Fold 0: train rows=64282, test rows=18162, test_pct=22.0%
Fold 1: train rows=66373, test rows=16071, test_pct=19.5%
Fold 2: train rows=66373, test rows=16071, test_pct=19.5%
Fold 3: train rows=66374, test rows=16070, test_pct=19.5%
Fold 4: train rows=66374, test rows=16070, test_pct=19.5%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Ans: ## Interpreting the fold-to-fold variability

Four candidate explanations for why the Random Forest underperforms the
baseline on Folds 2 and 4 were checked:

1. CTR variance — no monotonic relationship with performance (Fold 3 has the
   highest variance yet is a strong win).
2. Content-type composition — ruled out; every fold is 97-100% dominated by
   the same content type.
3. Word count — Fold 2 (the strongest underperformer, -9.2%) has a notably
   higher mean and much wider spread (mean 3,916, std 1,486) than every other
   fold (2,284-2,919 mean, 396-730 std). This is a real, standout difference
   and a plausible partial explanation for Fold 2 specifically — an unusually
   long-content, high-variance client mix may be harder for the model to
   generalize to from other folds' training data. However, it does not
   explain Fold 4 (the other underperformer), which has the LOWEST word count
   of any fold, not a high one.
4. Impression skew — high but not monotonic with performance; Fold 3 has by
   far the highest skew (27.0) yet is one of the strongest wins.

No single feature cleanly separates the winning folds (0, 1, 3) from the
losing folds (2, 4). Word count offers a plausible partial explanation for
Fold 2 alone. With only 5 non-overlapping client groups available, further
searching for a single unifying variable risks curve-fitting an explanation
to 5 data points rather than finding a genuine cause, so this investigation
is stopped here rather than continued indefinitely.

## Conclusion

The Random Forest achieves a 7.3% mean MAE improvement over the Week 4
baseline, but this masks real fold-to-fold variability (+20-23% on 3 folds,
-9% to -19% on 2). Of four checked candidate explanations, only word-count
distribution shows a plausible (though partial) link to one underperforming
fold; no single cause explains both. This could be reported as a genuine,
unresolved limitation of a small-N (5-fold) cross-client evaluation, not
papered over with a false unifying explanation. The model represents a real
but qualified improvement over the baseline not a uniform upgrade which
is consistent with this task's instruction to interpret errors honestly
rather than reward apparent average improvement alone.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def position_bucket(p):
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    return "21+"

model_df = df.copy()
model_df["position_bucket"] = model_df["avg_position_30d"].apply(position_bucket)
model_df["content_type"] = model_df["content_type"].fillna("unknown")
model_df["main_intent"]  = model_df["main_intent"].fillna("unknown")
model_df = pd.get_dummies(model_df, columns=["content_type", "main_intent"])

feature_cols = ["avg_position_30d", "impressions_30d", "sessions_30d",
                 "engaged_sessions_30d", "word_count"] + \
                [c for c in model_df.columns if c.startswith(("content_type_", "main_intent_"))]

X = model_df[feature_cols].fillna(0)
y = model_df["ctr_30d"]
groups = model_df["client_hash_id"]
pos_bucket = model_df["position_bucket"]

results = []
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, groups=groups)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    bucket_train, bucket_test = pos_bucket.iloc[train_idx], pos_bucket.iloc[test_idx]

    # --- BASELINE: predict the tier's mean CTR, computed from TRAIN fold only ---
    tier_means = y_train.groupby(bucket_train).mean()
    baseline_preds = bucket_test.map(tier_means).fillna(y_train.mean())  # fallback for any unseen bucket

    baseline_mae = mean_absolute_error(y_test, baseline_preds)
    baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_preds))

    # --- MODEL: Random Forest trained on TRAIN fold only ---
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_test)

    rf_mae = mean_absolute_error(y_test, rf_preds)
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

    results.append({
        "fold": fold, "n_test": len(test_idx),
        "baseline_mae": baseline_mae, "rf_mae": rf_mae,
        "baseline_rmse": baseline_rmse, "rf_rmse": rf_rmse,
        "mae_improvement_pct": (baseline_mae - rf_mae) / baseline_mae * 100
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

print("\nSummary (mean across folds)")
summary = pd.DataFrame({
    "Baseline (tier-mean CTR)": [results_df["baseline_mae"].mean(), results_df["baseline_rmse"].mean()],
    "Random Forest": [results_df["rf_mae"].mean(), results_df["rf_rmse"].mean()]
}, index=["MAE", "RMSE"])
print(summary)
print(f"\nMean MAE improvement over baseline: {results_df['mae_improvement_pct'].mean():.1f}%")

 fold  n_test  baseline_mae   rf_mae  baseline_rmse  rf_rmse  mae_improvement_pct
    0   18162      0.002213 0.001712       0.002871 0.002489            22.612001
    1   16071      0.002357 0.001840       0.002946 0.002717            21.939523
    2   16071      0.002574 0.002812       0.004465 0.004060            -9.228801
    3   16070      0.002665 0.002126       0.004739 0.003860            20.238877
    4   16070      0.002339 0.002788       0.003542 0.004188           -19.173030

Summary (mean across folds)
      Baseline (tier-mean CTR)  Random Forest
MAE                   0.002430       0.002255
RMSE                  0.003713       0.003463

Mean MAE improvement over baseline: 7.3%


In [8]:
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, groups=groups)):
    test_clients = model_df.iloc[test_idx]["client_hash_id"].unique()
    test_impressions = model_df.iloc[test_idx]["impressions_30d"]
    test_ctr = y.iloc[test_idx]
    print(f"Fold {fold}: {len(test_clients)} clients | "
          f"median impressions={test_impressions.median():.0f} | "
          f"CTR std={test_ctr.std():.5f} | CTR mean={test_ctr.mean():.5f}")

Fold 0: 1 clients | median impressions=1319 | CTR std=0.00287 | CTR mean=0.00232
Fold 1: 9 clients | median impressions=1058 | CTR std=0.00275 | CTR mean=0.00213
Fold 2: 10 clients | median impressions=1384 | CTR std=0.00453 | CTR mean=0.00303
Fold 3: 12 clients | median impressions=1203 | CTR std=0.00479 | CTR mean=0.00335
Fold 4: 10 clients | median impressions=757 | CTR std=0.00365 | CTR mean=0.00278


In [9]:
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, groups=groups)):
    test_ct = model_df.iloc[test_idx][[c for c in model_df.columns if c.startswith("content_type_")]].sum()
    dominant_type = test_ct.idxmax()
    dominant_pct = test_ct.max() / len(test_idx) * 100
    print(f"Fold {fold}: dominant content_type = {dominant_type} ({dominant_pct:.0f}% of test rows)")

Fold 0: dominant content_type = content_type_keyword article (100% of test rows)
Fold 1: dominant content_type = content_type_keyword article (97% of test rows)
Fold 2: dominant content_type = content_type_keyword article (98% of test rows)
Fold 3: dominant content_type = content_type_keyword article (98% of test rows)
Fold 4: dominant content_type = content_type_keyword article (99% of test rows)


In [10]:
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, groups=groups)):
    test_wc = model_df.iloc[test_idx]["word_count"]
    test_imp = model_df.iloc[test_idx]["impressions_30d"]
    print(f"Fold {fold}: word_count mean={test_wc.mean():.0f}, std={test_wc.std():.0f} | "
          f"impressions skew={test_imp.skew():.2f}")

Fold 0: word_count mean=2703, std=396 | impressions skew=6.84
Fold 1: word_count mean=2729, std=428 | impressions skew=9.53
Fold 2: word_count mean=3916, std=1486 | impressions skew=7.98
Fold 3: word_count mean=2919, std=730 | impressions skew=27.02
Fold 4: word_count mean=2284, std=638 | impressions skew=14.18


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Ans: **Where the model is wrong:** An initial check on Fold 0 was uninformative
(100% single-client test set, no valid comparison possible). Re-run on Fold
1 (9 test clients): client_9958f0a7ae1df715 is disproportionately represented
in the worst errors (6.3% of test rows, 30% of the 50 worst-error rows,
~5x over-representation) and remains the third-highest-error client (mean
abs_error 0.00266, n=1020) even after excluding unreliable tiny-n clients
(n<50). By contrast, the fold's dominant client (client_62f4a7e64f5e0096,
88.8% of test rows) shows the LOWEST mean error (0.00175) of any client with
meaningful volume — the model performs best on the bulk of the data and
worse on this smaller client.

Mechanism: client_9958f0a7ae1df715's pages sit predominantly at weaker
positions (mean avg_position_30d = 22.7, median = 14.2 — mostly the 11-20 and
21+ tiers) with moderate, right-skewed volume (median impressions_30d = 413,
mean pulled up to 822 by a handful of much larger pages, max 89,332). This is
the region of the position-CTR curve where CTR values are smallest and
noisiest (consistent with Section 1's finding that CTR differences compress
at lower positions), making precise prediction harder. Mean actual CTR
(0.00308) sits slightly below mean predicted CTR (0.00362) for this client,
indicating a modest, consistent over-prediction rather than random scatter.

A missing-word_count hypothesis, tested separately, did not hold: missing
word_count was actually less common among the worst 50 errors (44.0%) than
overall (58.8%), and content_type/main_intent contribute effectively nothing
to the model given their near-total class imbalance across folds.

**Bottom line:** the model's errors concentrate in lower-position,
lower-volume client segments, where CTR is inherently smaller and noisier,
and it slightly over-predicts CTR in that segment rather than under-
predicting it. This is a more precise and defensible characterization than
the aggregate metrics in Section 3 provide, and it directly explains part of
the fold-to-fold volatility already documented: folds whose test clients skew
toward this lower-position, lower-volume profile are the ones more likely to
see the Random Forest underperform the simple tier-average baseline, since a
tier-average is inherently more stable in a region where individual-page
prediction is hardest.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

# Refit on one representative fold (use fold 0 or refit on full data with a final holdout)
train_idx, test_idx = list(gkf.split(X, groups=groups))[0]
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

rf_final = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_final.fit(X_train, y_train)

perm = permutation_importance(rf_final, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

print(importance_df.head(10))

preds = rf_final.predict(X_test)
residuals = y_test.values - preds

error_df = model_df.iloc[test_idx][["content_hash_id", "client_hash_id", "avg_position_30d",
                                      "impressions_30d", "word_count"]].copy()
error_df["actual_ctr"] = y_test.values
error_df["predicted_ctr"] = preds
error_df["abs_error"] = np.abs(residuals)
error_df["signed_error"] = residuals  # positive = model under-predicted, negative = over-predicted

# The worst-predicted rows, both directions
print("--- Model most UNDER-predicted (actual >> predicted) ---")
print(error_df.sort_values("signed_error", ascending=False).head(10))

print("\n--- Model most OVER-predicted (predicted >> actual) ---")
print(error_df.sort_values("signed_error", ascending=True).head(10))

# Does error correlate with any feature? (e.g. does the model do worse on long content, low volume, etc.)
print("\n--- Correlation of abs_error with features ---")
print(error_df[["abs_error", "avg_position_30d", "impressions_30d", "word_count"]].corr()["abs_error"])

                            feature  importance_mean  importance_std
1                   impressions_30d     6.080902e-01    1.654453e-02
2                      sessions_30d     4.990218e-01    1.108964e-02
0                  avg_position_30d     1.908552e-01    5.896921e-03
3              engaged_sessions_30d     3.848625e-02    4.336009e-03
4                        word_count     8.284015e-03    1.625373e-03
5   content_type_comparison article     0.000000e+00    0.000000e+00
6       content_type_feedly article     0.000000e+00    0.000000e+00
7      content_type_keyword article     0.000000e+00    0.000000e+00
10         main_intent_navigational    -1.423110e-07    3.712752e-07
12              main_intent_unknown    -2.210867e-07    4.702387e-06
--- Model most UNDER-predicted (actual >> predicted) ---
                content_hash_id           client_hash_id  avg_position_30d  \
81132  content_eef81e0747996284  client_73cda7b4e4f265ea          3.193309   
66333  content_66547baf66929

In [13]:
# Confirm: is client_73cda7b4e4f265ea over-represented in the worst errors
# relative to its share of the test set overall?
client_share_in_test = (model_df.iloc[test_idx]["client_hash_id"] == "client_73cda7b4e4f265ea").mean()
print(f"client_73cda7b4e4f265ea share of Fold 0 test set: {client_share_in_test:.1%}")

worst_50 = error_df.reindex(error_df["abs_error"].sort_values(ascending=False).index).head(50)
client_share_in_worst = (worst_50["client_hash_id"] == "client_73cda7b4e4f265ea").mean()
print(f"client_73cda7b4e4f265ea share of the 50 worst-error rows: {client_share_in_worst:.1%}")

# Also check missing word_count rate specifically among the worst errors vs overall
missing_wc_overall = model_df.iloc[test_idx]["word_count"].isna().mean()
missing_wc_worst = worst_50["word_count"].isna().mean()
print(f"Missing word_count rate — overall test set: {missing_wc_overall:.1%}, worst 50 errors: {missing_wc_worst:.1%}")

client_73cda7b4e4f265ea share of Fold 0 test set: 100.0%
client_73cda7b4e4f265ea share of the 50 worst-error rows: 100.0%
Missing word_count rate — overall test set: 58.8%, worst 50 errors: 44.0%


In [14]:
# Rerun on Fold 1 instead — it has 9 test clients, so client-concentration is actually checkable
train_idx, test_idx = list(gkf.split(X, groups=groups))[1]
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

rf_f1 = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_f1.fit(X_train, y_train)
preds_f1 = rf_f1.predict(X_test)

error_df_f1 = model_df.iloc[test_idx][["content_hash_id", "client_hash_id", "avg_position_30d",
                                         "impressions_30d", "word_count"]].copy()
error_df_f1["actual_ctr"] = y_test.values
error_df_f1["predicted_ctr"] = preds_f1
error_df_f1["abs_error"] = np.abs(y_test.values - preds_f1)

# Now this check is actually meaningful — 9 clients to compare against each other
client_error = error_df_f1.groupby("client_hash_id")["abs_error"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print(client_error)

overall_share = error_df_f1["client_hash_id"].value_counts(normalize=True)
worst_50_f1 = error_df_f1.sort_values("abs_error", ascending=False).head(50)
worst_share = worst_50_f1["client_hash_id"].value_counts(normalize=True)
print("\nTop client share — overall vs worst-50:")
print(pd.DataFrame({"overall_share": overall_share, "worst_50_share": worst_share}).fillna(0))

                             mean  count
client_hash_id                          
client_59256b0571e0c970  0.003011      1
client_b77d0d5f08f05e64  0.002727     54
client_86ebc2f12c01f586  0.002666    113
client_9958f0a7ae1df715  0.002662   1020
client_810019792c9b8efc  0.002596      3
client_157ffe4d4a595515  0.002412    605
client_62f4a7e64f5e0096  0.001747  14269
client_2b4306c3ed003f01  0.001556      5
client_9d54435aabd95a6c  0.000240      1

Top client share — overall vs worst-50:
                         overall_share  worst_50_share
client_hash_id                                        
client_157ffe4d4a595515       0.037645            0.00
client_2b4306c3ed003f01       0.000311            0.00
client_59256b0571e0c970       0.000062            0.00
client_62f4a7e64f5e0096       0.887873            0.68
client_810019792c9b8efc       0.000187            0.00
client_86ebc2f12c01f586       0.007031            0.02
client_9958f0a7ae1df715       0.063468            0.30
client_9d5443

In [15]:
# Confirm client_9958f0a7ae1df715's over-representation is specific to it,
# not just "smaller clients generally error more" (which would be a volume artifact, not a client-specific one)
error_df_f1["client_row_count"] = error_df_f1.groupby("client_hash_id")["client_hash_id"].transform("count")

print(error_df_f1[["client_hash_id", "client_row_count", "abs_error"]]
      .groupby("client_hash_id")
      .agg(n=("abs_error","size"), mean_error=("abs_error","mean"))
      .query("n >= 50")  # drop unreliable tiny-n clients
      .sort_values("mean_error", ascending=False))

# What does client_9958f0a7ae1df715 look like on the features that matter most (per your importance ranking)?
suspect_client = error_df_f1[error_df_f1["client_hash_id"] == "client_9958f0a7ae1df715"]
print(suspect_client[["impressions_30d", "avg_position_30d", "actual_ctr", "predicted_ctr"]].describe())

                             n  mean_error
client_hash_id                            
client_b77d0d5f08f05e64     54    0.002727
client_86ebc2f12c01f586    113    0.002666
client_9958f0a7ae1df715   1020    0.002662
client_157ffe4d4a595515    605    0.002412
client_62f4a7e64f5e0096  14269    0.001747
       impressions_30d  avg_position_30d   actual_ctr  predicted_ctr
count      1020.000000       1020.000000  1020.000000    1020.000000
mean        822.082353         22.714194     0.003075       0.003623
std        3025.722185         18.797204     0.004468       0.003384
min         200.000000          2.984946     0.000000       0.000129
25%         274.750000          8.835098     0.000000       0.001368
50%         413.000000         14.151635     0.001433       0.002753
75%         743.500000         30.763712     0.004388       0.005124
max       89332.000000         87.711031     0.028986       0.032234


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.